# Mise en route sur Colab Pro (A100 40 GB), code sur Drive

**Runtime → Change runtime type → A100 GPU** avant de commencer.

Le dossier du dépôt va à la racine du Drive, puis les cellules s'exécutent dans
l'ordre. Une seule ligne à éditer, cellule 2 : le chemin de ce dossier.

### Où va quoi, et pourquoi

Monter Drive et tout y faire tourner ne marche pas : les trois sortes de fichiers de ce banc
veulent trois endroits différents.

| | Emplacement | Pourquoi |
|---|---|---|
| **Code** | Drive | lu une fois à l'import ; FUSE convient, et une modif sur Drive prend effet au run suivant sans rien copier |
| **Données** | VM locale | CIFAR/STL sont relus à **chaque époque** ; à travers FUSE le GPU attend le réseau. Les re-télécharger coûte une minute par session, bien moins que la perte de débit |
| **Résultats** | local, synchronisé vers Drive | le socle `fsync` le CSV à **chaque ligne** et écrit ses checkpoints par `os.replace()`. Le fsync par ligne sur FUSE fait caler l'entraînement, et `os.replace()` n'est pas fiablement atomique à travers un montage FUSE, ce qui annule tout l'intérêt de l'écrire ainsi |

`colab_run.py` fait ce partage tout seul. Et surtout, il **rapatrie les résultats depuis
Drive au démarrage** : c'est ce qui permet à `--resume` de reprendre après une session
coupée, au lieu de repartir de zéro à chaque fois.

In [ ]:
# 1. Vérifier le GPU AVANT toute chose.
#    Un A100 est attendu ; sur T4 les coûts du README sont à multiplier par ~8.
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'{p.name}  {p.total_memory/1e9:.0f} GB')
    if 'A100' not in p.name:
        print('\n*** Ce n\'est pas un A100. Les coûts du README ne tiennent plus. ***')
else:
    print('\n*** Aucun GPU. Runtime -> Change runtime type -> A100. ***')

In [ ]:
# 2. Monter Drive.  SEULE LIGNE À ÉDITER : le chemin du dossier sur Drive.
DRIVE_DIR = '/content/drive/MyDrive/ubergang_xp'
LOCAL_DIR = '/content/ubergang_local'      # données + résultats en cours, sur la VM

from google.colab import drive
drive.mount('/content/drive')

import os, sys
assert os.path.isdir(DRIVE_DIR), (
    f"{DRIVE_DIR} introuvable. Le dossier du dépôt doit être sur le Drive, "
    f"ou corrige DRIVE_DIR ci-dessus.")
for need in ('lib/harness.py', 'colab_run.py', 'aggregate.py', 'xp'):
    assert os.path.exists(os.path.join(DRIVE_DIR, need)), f'manquant sur Drive : {need}'

# Le code tourne DEPUIS Drive : pas de copie, une modif là-bas prend effet tout de suite.
os.chdir(DRIVE_DIR)
sys.path.insert(0, DRIVE_DIR)
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'   # pas de __pycache__ sur FUSE
os.makedirs(LOCAL_DIR, exist_ok=True)
print('cwd     =', os.getcwd())
print('contenu =', sorted(os.listdir('.')))

In [ ]:
# 3. Dépendances. Sur Colab tout est déjà là : cette cellule ne devrait rien installer.
!pip install -q -r requirements.txt
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__)

In [ ]:
# 4. Auto-test du socle. DOIT passer avant de dépenser la moindre minute de GPU. ~3 s.
#    Il vérifie entre autres l'invariance O(d) de la cible relationnelle, la NON-invariance
#    de la tête à prototypes, le refus de vicreg_reg sur un z normalisé, le gel effectif des
#    têtes du panel, et la reprise du Run.
!python lib/harness.py

In [ ]:
# 5. Smoke de bout en bout, les six groupes. Valide le PIPELINE, jamais une thèse.
#    Quelques minutes. Un smoke qui casse ici t'évite de le découvrir à l'heure 40.
#    Les sorties smoke vont dans le dossier LOCAL : elles ne polluent pas results/ sur Drive.
import subprocess, glob, time, os
smoke_out = os.path.join(LOCAL_DIR, 'smoke')
for f in sorted(glob.glob('xp/xp_*.py')):
    t0 = time.time()
    r = subprocess.run(
        ['python', f, '--smoke', '--data-root', os.path.join(LOCAL_DIR, 'data'),
         '--outdir', os.path.join(smoke_out, os.path.basename(f)[3:4])],
        capture_output=True, text=True)
    tail = (r.stdout or r.stderr).strip().splitlines()[-1:] or ['(pas de sortie)']
    print(f"{'OK   ' if r.returncode == 0 else 'ÉCHEC'} {f:44s} {time.time()-t0:6.0f}s  {tail[0][:66]}")

## 6. Lancer une expérience

Toujours **via `colab_run.py`** : il restaure les résultats depuis Drive (donc `--resume`
reprend d'une session à l'autre), place les données en local, et resynchronise vers Drive
toutes les 2 minutes **et à la sortie**, y compris sur Ctrl-C ou sur plantage.

Ordre dans lequel je les lance (coûts dans `README.md`) :

| # | Commande | Coût | Pourquoi |
|---|---|---|---|
| 1 | `python xp/xp_A_diagnostics.py --exp a1` | 20 s | exact, binaire |
| 2 | `python xp/xp_A_diagnostics.py` | ~0,3 h | établit la réserve sur T2 |
| 3 | `python xp/xp_C_panel.py --exp c1` | ~5 h | binaire : peut donner tort à T3 en quelques heures |
| 4 | `python xp/xp_E_router.py --stage e0` | ~1 h | GO/NO-GO : décide si E1 mérite une semaine |
| 5 | `python xp/xp_D_augmentations.py --exp D1` | ~2-3 h | T5 : T dégénère vers l'identité |
| 6 | `python xp/xp_D_augmentations.py --exp D2` | ~2-4 h | T6 : asymétrie de T |
| 7 | `python xp/xp_B_relational_vs_prototype.py --all` | **~47 h** | la principale, à fractionner |
| 8 | `python xp/xp_F_frozen_substrate.py` | ~18-20 h | T7 et T8 |

Relancer exactement la même ligne après une coupure reprend où ça s'était arrêté.

In [ ]:
# 7. L'expérience. Édite CMD, puis exécute.
CMD = 'python xp/xp_A_diagnostics.py --exp a1'

!python colab_run.py --drive "$DRIVE_DIR" --local "$LOCAL_DIR" --sync-every 120 -- {CMD}

In [ ]:
# 8. Synthèse, lue directement depuis les résultats sur Drive.
#    Exclut automatiquement tout summary de mode --smoke, et downgrade en 'insuffisant'
#    tout verdict appuyé sur moins de 3 graines.
!python aggregate.py --results "$DRIVE_DIR/results" --csv "$DRIVE_DIR/results/synthese.csv"

In [ ]:
# 9. Filet de sécurité : synchronisation manuelle vers Drive.
#    colab_run.py le fait déjà tout seul ; celle-ci sert après une commande lancée à la
#    main sans passer par lui, ou pour forcer la synchro avant de fermer l'onglet.
!mkdir -p "$DRIVE_DIR/results" && cp -ru "$LOCAL_DIR/results/." "$DRIVE_DIR/results/" 2>/dev/null; \
 du -sh "$DRIVE_DIR/results" && echo 'résultats synchronisés vers Drive'